In [1]:
from matplotlib import pyplot as plt
import pandas
import tensorflow as tf
import tiktoken
import numpy
import json
from chapter02.dataset import SpamDataset
import keras


E:\Projects\Python\Machine-Learning\.venv\Lib\site-packages\tensorflow\python\keras\engine\training_arrays_v1.py:37: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.3)
  from scipy.sparse import issparse  # pylint: disable=g-import-not-at-top


Load dataset

In [2]:
batch_size = 8
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


In [3]:
train_dataset = SpamDataset(
    '../data/sms_spam_collection/train.csv',
    tokenizer=tokenizer,
    max_length=None,
    batch_size=batch_size
)

print(train_dataset.max_length)

120


In [4]:
val_dataset = SpamDataset(
    '../data/sms_spam_collection/validation.csv',
    tokenizer=tokenizer,
    max_length=train_dataset.max_length,
    batch_size=batch_size
)

test_dataset = SpamDataset(
    "../data/sms_spam_collection/test.csv",
    tokenizer=tokenizer,
    max_length=train_dataset.max_length,
    batch_size=batch_size
)

As a verification step, we iterate through the data loaders and ensure that the batches contain 8 training examples each, where each training example consists of 120 tokens

In [5]:
print("Train dataset:")

batch0 = train_dataset.__getitem__(0)
print("Input batch dimensions:", batch0[0].shape)
print("Label batch dimensions", batch0[1].shape)

Train dataset:
Input batch dimensions: (8, 120)
Label batch dimensions (8,)


Lastly, let's print the total number of batches in each dataset

In [6]:
print(f"{train_dataset.__len__()} training batches")
print(f"{val_dataset.__len__()} validation batches")
print(f"{test_dataset.__len__()} test batches")

130 training batches
18 validation batches
37 test batches


## 6.6 Initializing a model with pretrained weights

In [7]:
# import config
config = json.load(open('../GPT_CONFIG_124M.json', mode='r', encoding='UTF-8'))

assert train_dataset.max_length <= config.get('context_length'), (
    f"Dataset length {train_dataset.max_length} exceeds model's context "
    f"length {config['context_length']}. Reinitialize data sets with "
    f"`max_length={config['context_length']}`"
)

config.update(config['model_configs'][config.get('CHOOSE_MODEL')])
print(config)

{'CHOOSE_MODEL': 'gpt2-small (124M)', 'INPUT_PROMPT': 'Every effort moves', 'vocab_size': 50257, 'context_length': 1024, 'drop_rate': 0.0, 'batch_size': 16, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_embed': 0.1, 'dropout_mha': 0.1, 'dropout_after_mha': 0.1, 'dropout_feedforward': 0.1, 'qkv_bias': True, 'epochs': 20, 'model_configs': {'gpt2-small (124M)': {'emb_dim': 768, 'n_layers': 12, 'n_heads': 12}, 'gpt2-medium (355M)': {'emb_dim': 1024, 'n_layers': 24, 'n_heads': 16}, 'gpt2-large (774M)': {'emb_dim': 1280, 'n_layers': 36, 'n_heads': 20}, 'gpt2-xl (1558M)': {'emb_dim': 1600, 'n_layers': 48, 'n_heads': 25}}}


In [8]:
from chapter04_LLM_arch.GPTArchitecture import GPTModel
from gpt_download import download_and_load_gpt2
from load_weights import load_weights_into_gpt
from chapter04_LLM_arch.generate_text_simple import generate_text_simple

# build the model
dummp_inputs = tf.zeros(shape=(1, config.get('context_length')), dtype=tf.int32)
model = GPTModel(config)
model(dummp_inputs)

model_size = config.get('CHOOSE_MODEL').split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="../save_weights/gpt2")
load_weights_into_gpt(model, params)


File already exists and is up-to-date: ../save_weights/gpt2\124M\checkpoint
File already exists and is up-to-date: ../save_weights/gpt2\124M\encoder.json
File already exists and is up-to-date: ../save_weights/gpt2\124M\hparams.json
File already exists and is up-to-date: ../save_weights/gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: ../save_weights/gpt2\124M\model.ckpt.index
File already exists and is up-to-date: ../save_weights/gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: ../save_weights/gpt2\124M\vocab.bpe


Before we finetune the model as a classifier, let's see if the model can perhaps already classify spam messages via prompting

In [9]:
from chapter05_pretraining.text_id_convertion import text_to_id, id_to_text

text_example = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':"
    " 'You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award.'"
)

token_ids  = generate_text_simple(model,
                     idx=text_to_id(text_example, tokenizer, model),
                     max_new_tokens=30,
                     context_size=config.get('context_length'))

print(f'Inputs: {text_example}\nWhether if it is spam: {id_to_text(token_ids, tokenizer, model)}')

Inputs: Is the following text 'spam'? Answer with 'yes' or 'no': 'You are a winner you have been specially selected to receive $1000 cash or a $2000 award.'
Whether if it is spam: Is the following text 'spam'? Answer with 'yes' or 'no': 'You are a winner you have been specially selected to receive $1000 cash or a $2000 award.'

The following text 'spam'? Answer with 'yes' or 'no': 'You are a winner you have been specially selected to receive


As we can see, the model is not very good at following instructions<br>
This is expected, since it has only been pretrained and not instruction-finetuned (instruction finetuning will be covered in the next chapter)

## 6.5 Adding a classification head

In [11]:
# Let's take a look at the model architecture first
model.summary()

Model: "gpt_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (1, 1024, 768)         │    38,597,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (1024, 768)            │       786,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ ?                      │     8,136,448 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ ?                      │     8,136,448 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ ?                      │     8,136,448 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ ?                      │     8,136,448 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_4             │ ?                      │     8,136,448 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_5             │ ?                      │     8,136,448 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_6             │ ?                      │     8,136,448 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_7             │ ?                      │     8,136,448 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_8             │ ?                      │     8,136,448 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_9             │ ?                      │     8,136,448 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_10            │ ?                      │     8,136,448 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_11            │ ?                      │     8,136,448 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_norm_24 (LayerNorm)       │ ?                      │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_72 (Dense)                │ (1, 1024, 50257)       │    38,647,633 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 175,670,353 (670.13 MB)

 Trainable params: 163,087,441 (622.13 MB)

 Non-trainable params: 12,582,912 (48.00 MB)